# Introduction

This notebook implements a convolutional neural network (CNN) to predict car brand and model using structured input features.  
The workflow includes:

- data loading and preprocessing
- feature engineering
- building and training a CNN model
- evaluating performance for brand and model predictions
- saving and inference for new samples

Goal: demonstrate an end-to-end ML pipeline where CNN learns to classify brand/model from available attributes.

In [28]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('dataset.csv')

# Display the first few rows of the dataset
print(df.head())

print(df.info())

  year\tmake\tmodel\ttrim\tbody\ttransmission\tvin\tstate\tcondition\todometer\tcolor\tinterior\tseller\tmmr\tsellingprice\tsaledate
0  2015\tKia\tSorento\tLX\tSUV\tautomatic\t5xyktc...                                                                                
1  2015\tKia\tSorento\tLX\tSUV\tautomatic\t5xyktc...                                                                                
2  2014\tBMW\t3 Series\t328i SULEV\tSedan\tautoma...                                                                                
3  2015\tVolvo\tS60\tT5\tSedan\tautomatic\tyv1612...                                                                                
4  2014\tBMW\t6 Series Gran Coupe\t650i\tSedan\ta...                                                                                
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 558837 entries, 0 to 558836
Data columns (total 1 columns):
 #   Column                                                                                 

In [ ]:
# Data Cleaning

Data cleaning is a crucial early step in the pipeline.  
It ensures the dataset is reliable and consistent before feature engineering or model training.

- Split raw combined string rows into separate logically named columns (`year`, `make`, `model`, etc.).
- Convert types:
    - categorical text to uppercase (standardization)
    - numeric fields (`year`, `odometer`, `mmr`, `sellingprice`, `condition`) to numeric type
    - date strings (`saledate`) to proper `datetime`
- Normalize continuous variables (`condition`) with MinMax scaling so ranges align for ML and comparison.
- Handle missing and invalid values:
    - fill missing or invalid condition with “Unknown” after normalization
    - drop rows where core numeric labels (`mmr`, `sellingprice`, `year`) are missing

Goal:
- eliminate format inconsistencies
- enforce accurate types for calculations
- reduce noise from missing/invalid data
- make downstream analysis/ML resilient and interpretable.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Create new features based on existing ones, and drop original column
split_columns = df['year\tmake\tmodel\ttrim\tbody\ttransmission\tvin\tstate\tcondition\todometer\tcolor\tinterior\tseller\tmmr\tsellingprice\tsaledate'].str.split('\t', expand=True)
split_columns.columns = ['year','make', 'model', 'trim', 'body', 'transmission', 'vin', 'state', 'condition', 'odometer', 'color', 'interior', 'seller', 'mmr', 'sellingprice', 'saledate']
df_clean = pd.concat([df, split_columns], axis=1)
df_clean.drop(df_clean.columns[0], axis=1, inplace=True)

# Convert all string columns to uppercase
df_upper_case = df_clean.copy()
df_upper_case = df_upper_case.applymap(lambda x: x.upper() if isinstance(x, str) else x)

#Convert saledate to datetime format
df_upper_case['saledate'] = pd.to_datetime(df_upper_case['saledate'], errors='coerce')

# Convert numeric columns to appropriate data types
numeric_columns = ['year', 'odometer', 'mmr', 'sellingprice', 'condition']
for col in numeric_columns:
    df_upper_case[col] = pd.to_numeric(df_upper_case[col], errors='coerce')

# Apply Min-Max scaling to the 'condition' column
scaler = MinMaxScaler()
df_scaled = df_upper_case.copy()
df_scaled['condition_normalized'] = scaler.fit_transform(df_scaled[['condition']])
df_scaled.drop('condition', axis=1, inplace=True)

# Null values in condition as Unknown
df_scaled['condition_normalized'].fillna('Unknown', inplace=True)

# Drop instances where mmr, selingprice, odometer or year is null
df_no_nulls = df_scaled.copy()
df_no_nulls.dropna(subset=['mmr', 'sellingprice', 'odometer', 'year'], inplace=True)

print(f'Unique values in the "body" column: {df_no_nulls["body"].unique()}')
print(f'Null values in the dataset:\n{df_no_nulls.isnull().sum()}')





Unique values in the "body" column: ['SUV' 'SEDAN' 'CONVERTIBLE' 'COUPE' 'WAGON' 'HATCHBACK' 'CREW CAB'
 'G COUPE' 'G SEDAN' 'ELANTRA COUPE' 'GENESIS COUPE' 'MINIVAN' '' 'VAN'
 'DOUBLE CAB' 'CREWMAX CAB' 'ACCESS CAB' 'KING CAB' 'SUPERCREW'
 'CTS COUPE' 'EXTENDED CAB' 'E-SERIES VAN' 'SUPERCAB' 'REGULAR CAB'
 'G CONVERTIBLE' 'KOUP' 'QUAD CAB' 'CTS-V COUPE' 'G37 CONVERTIBLE'
 'CLUB CAB' 'XTRACAB' 'Q60 CONVERTIBLE' 'CTS WAGON' 'G37 COUPE' 'MEGA CAB'
 'CAB PLUS 4' 'Q60 COUPE' 'CAB PLUS' 'BEETLE CONVERTIBLE'
 'TSX SPORT WAGON' 'PROMASTER CARGO VAN' 'GRANTURISMO CONVERTIBLE'
 'CTS-V WAGON' 'RAM VAN' 'TRANSIT VAN' 'REGULAR-CAB']
Null values in the dataset:
year                     0
make                     0
model                    0
trim                     0
body                     0
transmission             0
vin                      0
state                    0
odometer                94
color                    0
interior                 0
seller                   0
mmr                